# Power Dissipation Index (PDI) demo (EXPERIMENTAL)

This notebook demonstrates the **experimental** `tcpyPI.power_dissipation_index()` utility using:

- **Observed storm intensity** from IBTrACS (`data/IBTrACS.NA.v04r01.nc`)
- A worked example storm: **Hurricane Milton (2024)**

PDI is defined as:

$$\mathrm{PDI} = \int V_{\max}(t)^3\,dt$$

## Important scientific caveats

- **PDI is a track/intensity-history metric**. In standard usage, $V_{\max}(t)$ is realized storm intensity (best-track, model TC tracker, etc.).
- In this notebook, PDI is computed from **IBTrACS `usa_wind`** (knots).
- PDI is sensitive to **sampling cadence and missing data**. Here we infer $\Delta t$ from the IBTrACS timestamps and use `nan_policy="omit"`.


## 0) Setup

This repo uses a `src/` layout. This cell locates the repository root (by searching for `src/tcpyPI`) and adds `src/` to the import path so imports resolve without installation.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
repo_root = None
for cand in [here] + list(here.parents):
    if (cand / "src" / "tcpyPI").is_dir():
        repo_root = cand
        break
if repo_root is None:
    raise RuntimeError(
        "Could not locate repo root (expected src/tcpyPI). "
        "Run from within the repo or install with `pip install -e .`."
    )

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

import tcpyPI

print("tcpyPI from:", tcpyPI.__file__)


## 1) Load IBTrACS and select Hurricane Milton (2024)

This demo expects `data/IBTrACS.NA.v04r01.nc` to be present (North Atlantic basin, v04r01).

In [ ]:
ib = xr.open_dataset(repo_root / "data" / "IBTrACS.NA.v04r01.nc")

# Identify the storm by name + season.
names = ib["name"].astype(str)
seasons = ib["season"].astype(int)
mask = (names == "MILTON") & (seasons == 2024)
storm_idx = np.where(mask.values)[0]

if storm_idx.size != 1:
    raise ValueError(f"Expected exactly 1 Milton(2024) storm, found {storm_idx.size}.")

s = int(storm_idx[0])
print("IBTrACS sid:", ib["sid"].isel(storm=s).values)
print("Season:", int(ib["season"].isel(storm=s).values))


## 2) Extract track, intensity, and compute PDI

We use IBTrACS `usa_wind` (typically ATCF best-track intensity) in **knots**.

For the discrete time series, we approximate the integral with a Riemann sum:

$$\mathrm{PDI} pprox \sum_i V_{\max,i}^3\,\Delta t_i$$

where $\Delta t_i$ is inferred from time differences.

In [ ]:
time = ib["time"].isel(storm=s)
lat = ib["lat"].isel(storm=s)
lon = ib["lon"].isel(storm=s)
wind_kt = ib["usa_wind"].isel(storm=s)
status = ib["usa_status"].isel(storm=s).astype(str)

valid = np.isfinite(lat.values) & np.isfinite(lon.values) & np.isfinite(wind_kt.values)

t = time.values[valid]
lat = lat.values[valid].astype(float)
lon = lon.values[valid].astype(float)
vmax_kt = wind_kt.values[valid].astype(float)
status = status.values[valid]

# IBTrACS times sometimes have tiny offsets; round to the nearest minute for clean plotting.
t = t.astype('datetime64[m]')

# Build dt array (hours) aligned with vmax.
# Use forward differences, with the last step equal to the previous dt.
dt_h = np.diff(t).astype('timedelta64[m]').astype(float) / 60.0
if dt_h.size == 0:
    raise ValueError('Storm has too few valid points to compute PDI.')
dt_h = np.concatenate([dt_h[:1], dt_h])
dt_h[-1] = dt_h[-2]

print('Valid fixes:', t.size)
print('Time range:', t[0], 'to', t[-1])
print('IBTrACS usa_wind units:', wind_kt.attrs.get('units', 'kts (assumed)'))
print('Status mix:', sorted(set(status.tolist())))
print('Typical dt (h):', float(np.nanmedian(dt_h)))

pdi_si = tcpyPI.power_dissipation_index(vmax_kt, dt_h, wind_units='kt', dt_units='h', nan_policy='omit')
pdi_scaled = tcpyPI.power_dissipation_index(vmax_kt, dt_h, wind_units='kt', dt_units='h', formulation='e05_1e11', nan_policy='omit')

print('PDI (e05_si)  :', pdi_si)
print('PDI (e05_1e11):', pdi_scaled)


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax[0].plot(lon, lat, marker='o')
ax[0].set_title('Hurricane Milton (2024) track (IBTrACS)')
ax[0].set_xlabel('lon')
ax[0].set_ylabel('lat')
ax[0].grid(True)

ax[1].plot(t, vmax_kt)
ax[1].set_title('Intensity time series (IBTrACS usa_wind)')
ax[1].set_ylabel('kt')
ax[1].grid(True)

plt.tight_layout()
plt.show()


## Notes on options

`power_dissipation_index` supports:
- `wind_units`: `"m/s"` or `"kt"`
- `dt_units`: `"s"` or `"h"`
- `formulation`: `"e05_si"` (raw) or `"e05_1e11"` (scaled)
- `nan_policy`: `"propagate"` (default) or `"omit"`

`dt` can be a scalar (constant timestep) or an array broadcastable to `vmax` (variable timestep).